In [2]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erfc
import csv
import time
import os


###################################################### PARAMETERS #############################################
path= "./custom/"
p_radius = 5
p_totalTime = 2.0
p_stepTime = 10**-4
p_diffusionCoef = 79.4
p_distance = 12.5
p_nofMolecules = 10**6


normalization_parameter = 10




p_variablesx = [float(i)/5 for i in range(1,3)]
p_variablesy = [float(i)/5 for i in range(1,3)]
p_variablesz = [float(i)/5 for i in range(3,4)]
override = False

In [3]:

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("cuda" if torch.cuda.is_available() else "cpu")

# Convert parameters to tensors
radius = torch.tensor(p_radius, device=device)
totalTime = torch.tensor(p_totalTime, device=device)
stepTime = torch.tensor(p_stepTime, device=device)
diffusionCoef = torch.tensor(p_diffusionCoef, device=device)
distance = torch.tensor(p_distance, device=device)
nofMolecules = p_nofMolecules
variablesx = torch.tensor(p_variablesx)
variablesy = torch.tensor(p_variablesy)
variablesz = torch.tensor(p_variablesz)

filelist=[]

cuda


In [4]:
def Fhit_function(radius, distance, diffusionCoef, stepTime):
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * stepTime))


def binomialP(radius, distance, diffusionCoef , stepTime , nofMolecules, time_values):
    Fhit_values = np.array([Fhit_function(radius, distance, diffusionCoef, t) for t in time_values])
    Fhit_shifted = np.concatenate(([0], Fhit_values[:-1]))
    return (Fhit_values - Fhit_shifted)

def binomial(radius, distance, diffusionCoef, stepTime, nofMolecules, time_values):
    # Calculate the probabilities for each time step
    probabilities = binomialP(radius, distance, diffusionCoef, stepTime, nofMolecules, time_values)

    # Simulate the binomial random variable for each time step
    binomial_results = np.random.binomial(nofMolecules, probabilities)

    return binomial_results

In [5]:
def save_binomial_data_as_csv(directory_name,filename):
    # Check if the directory exists
    
    output_directory =  directory_name 
    if not os.path.exists(output_directory):
        # If the directory does not exist, create it
        os.makedirs(output_directory)
       
    csv_file_path = output_directory + filename+".csv"
    times_for_theoretical = np.linspace(p_stepTime, p_totalTime, int(p_totalTime/p_stepTime))
    theoretical_line = binomial(p_radius, p_distance-p_radius, p_diffusionCoef, p_stepTime, int(p_nofMolecules), times_for_theoretical)
    combined_data = np.column_stack((times_for_theoretical, theoretical_line))
    
    np.savetxt(csv_file_path, combined_data, delimiter=',', header='time,combined_molecules', comments='')

    


In [7]:
import torch
import os
import csv
from itertools import product
directory_list=[]


def experimenttotalsum_optimized2(name,radius, totalTime, stepTime, diffusionCoef, distance, nofMolecules, device):
    nofSteps = int(totalTime / stepTime) + 1

    # Create a directory name using all alfa_values
  

    # Initialize molecule positions and states
    positions = torch.zeros((nofMolecules, 3), device=device)
    positions[:, 2] = distance
    state = torch.ones(nofMolecules, device=device)

    # Precompute standard deviation for Gaussian noise
    sigma = torch.sqrt(2 * diffusionCoef * stepTime)
    sum_value = 0
    nofMolecules_list = torch.zeros(nofSteps, device=device)
    time_list = torch.arange(0, totalTime.item() + stepTime.item(), stepTime.item(), device=device)

    # Create a filename using alfa_value

    filename = path +"Exp " +str(name)+ ".csv"
    if os.path.exists(filename) and not override:
        print(f"File {filename} already exists. Skipping...")
        return 

    # Open a file to write the results
    with open(filename, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["Time", "Number of Molecules"])

        for step in range(nofSteps):
            # Generate Gaussian noise for molecule movement
            noise = torch.randn_like(positions) * sigma
            positions += noise
            inside_mask = (positions.pow(2).sum(dim=1) <= radius**2) & (state > 0)
            sum_value += inside_mask.float().sum()
            state[inside_mask] = -1.0
            nofMolecules_list[step] = inside_mask.float().sum()
            writer.writerow([time_list[step].item(), nofMolecules_list[step].item()])

        # Reset positions and state for the next iteration
        positions.zero_()
        positions[:, 2] = distance
        state.fill_(1.0)

   
    print("Total Number of Molecules inside Sphere:", sum_value)


In [8]:

############################################################## MAIN ##########################################################


# Call the function
start_time = time.time()
for i in range(1):
    sum_value = experimenttotalsum_optimized2(i,radius, totalTime, stepTime, diffusionCoef, distance, nofMolecules, device)

end_time = time.time()

execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")
# Print results
print("Total Number of Molecules inside Sphere:", sum_value)

File ./custom/Exp 0.csv already exists. Skipping...
Execution time: 0.032561 seconds
Total Number of Molecules inside Sphere: None


In [12]:
save_binomial_data_as_csv("./","binomial")